In [ ]:
from deepfmkit.experiments import Experiment
from deepfmkit.waveforms import shd
import numpy as np

from deepfmkit.plotting import default_rc, cmap_parula_r
import matplotlib.pyplot as plt
plt.rcParams.update(default_rc)

In [ ]:
from deepfmkit.factories import StandardDFMIExperimentFactory

factory = StandardDFMIExperimentFactory(waveform_function=shd, opd_main=0.05)

experiment = Experiment(description="2nd Harmonic Distortion", seed=29)
experiment.set_config_factory(factory)
experiment.n_trials = 500
experiment.f_samp = 200e3
experiment.n_fit_buffers_per_trial = 1

axis = np.linspace(0.0, 0.0006, 10)
experiment.add_axis("distortion_amp", axis)

experiment.set_static(
    {
        "m_main": 6.0,
    }
)

experiment.add_stochastic_variable(
    "waveform_kwargs",
    lambda dist_amp: {
        "distortion_amp": dist_amp,
        "distortion_phase": np.random.uniform(-np.pi, np.pi),
    },
    depends_on="distortion_amp",
)
experiment.add_stochastic_variable("phi", lambda: np.random.uniform(-np.pi, np.pi))

experiment.add_analysis(
    name="NLS_Fit",
    fitter_method="nls",
    result_cols=["m"],
    fitter_kwargs={
        "ndata": 30,
    },
)

print(f"Starting experiment: '{experiment.description}'...")
experiment.results = experiment.run()
print("Experiment completed.")

In [ ]:
results = experiment.results
x_axis = results["axes"]["distortion_amp"]

fig1, ax = plt.subplots(figsize=(3.375, 2), dpi=150)
mean_m = results["NLS_Fit"]["m"]["mean"]
std_m = results["NLS_Fit"]["m"]["std"]

ax.errorbar(
    x_axis * 100,
    mean_m,
    yerr=std_m,
    fmt="o-",
    lw=2,
    capsize=3,
    markersize=5,
    c="k",
    zorder=-1,
)
ax.set_xlabel("Distortion amplitude (%)")
ax.set_ylabel(r"Estimate of $m$")
ax.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
factory = StandardDFMIExperimentFactory(waveform_function=shd, opd_main=0.05)

experiment = Experiment(description="2nd Harmonic Distortion", seed=42)
experiment.set_config_factory(factory)
experiment.n_trials = 10
experiment.f_samp = 200e3
experiment.n_fit_buffers_per_trial = 10

experiment.add_axis("m_main", np.linspace(3.0, 30.0, 100))
experiment.add_axis("distortion_amp", np.linspace(0.0, 0.2, 100))

experiment.add_stochastic_variable(
    "waveform_kwargs",
    lambda dist_amp: {
        "distortion_amp": dist_amp,
        "distortion_phase": np.random.uniform(-np.pi, np.pi),
    },
    depends_on="distortion_amp",
)

experiment.add_stochastic_variable("phi", lambda: np.random.uniform(-np.pi, np.pi))

experiment.add_analysis(
    name="NLS_Fit",
    fitter_method="nls",
    result_cols=["m", "phi"],
    fitter_kwargs={
        "ndata": 30,
    },
)

print(f"Starting experiment: '{experiment.description}'...")
experiment.results = experiment.run(filename="1.1_harmonic-distortion-bias.data")
print("Experiment completed.")

In [ ]:
experiment = Experiment(filename="1.1_harmonic-distortion-bias.data")
results = experiment.results
threshold = 0.3

m_true_axis = results["axes"]["m_main"]
distortion_amp_axis = results["axes"]["distortion_amp"]
m_estimated_all = results["NLS_Fit"]["m"]["all_trials"]
m_true_broadcast = m_true_axis[:, np.newaxis, np.newaxis]
bias_all_trials = m_estimated_all - m_true_broadcast
mean_bias = np.mean(np.abs(bias_all_trials), axis=-1)
worst_case_bias = np.max(np.abs(bias_all_trials), axis=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.875, 2.5), dpi=150, sharey=True)

pcm1 = ax1.pcolormesh(
    m_true_axis,
    distortion_amp_axis * 100,
    mean_bias.T,
    cmap=cmap_parula_r,
    shading="nearest",
    vmin=0,
    vmax=threshold,
)
ax1.set_title(r"Mean bias: $|E\,[\hat{m}] - m|$")
ax1.set_xlabel("True modulation depth $m$ (rad)")
ax1.set_ylabel("Distortion amplitude (%)")

pcm2 = ax2.pcolormesh(
    m_true_axis,
    distortion_amp_axis * 100,
    worst_case_bias.T,
    cmap=cmap_parula_r,
    shading="nearest",
    vmin=0,
    vmax=threshold,
)
ax2.set_title(r"Worst-case bias: max$| \hat{m} - m|$")
ax2.set_xlabel("True modulation depth, $m$ (rad)")

fig.subplots_adjust(right=0.85)
cbar_ax = fig.add_axes([0.87, 0.15, 0.03, 0.7])
cbar = fig.colorbar(pcm2, cax=cbar_ax)
cbar.set_label("Bias in $m$ (rad)")

plt.show()

In [ ]:
factory = StandardDFMIExperimentFactory(waveform_function=shd, opd_main=0.05)

experiment = Experiment(description="2nd Harmonic Distortion", seed=42)
experiment.set_config_factory(factory)
experiment.n_trials = 20
experiment.f_samp = 200e3
experiment.n_fit_buffers_per_trial = 10

experiment.add_axis("m_main", np.linspace(16.0, 16.025, 100))
experiment.add_axis("distortion_amp", np.linspace(0.0, 0.0006, 100))

experiment.add_stochastic_variable(
    "waveform_kwargs",
    lambda dist_amp: {
        "distortion_amp": dist_amp,
        "distortion_phase": np.random.uniform(-np.pi, np.pi),
    },
    depends_on="distortion_amp",
)

experiment.add_stochastic_variable("phi", lambda: np.random.uniform(-np.pi, np.pi))

experiment.add_analysis(
    name="NLS_Fit",
    fitter_method="nls",
    result_cols=["m", "phi"],
    fitter_kwargs={
        "ndata": 30,
    },
)

print(f"Starting experiment: '{experiment.description}'...")
experiment.results = experiment.run(filename="1.1_harmonic-distortion-bias-02.data")
print("Experiment completed.")

In [ ]:
experiment = Experiment(filename='1.1_harmonic-distortion-bias-02.data')
results = experiment.results
threshold = 1e-5

m_true_axis = results["axes"]["m_main"]
distortion_amp_axis = results["axes"]["distortion_amp"]
m_estimated_all = results["NLS_Fit"]["m"]["all_trials"]
m_true_broadcast = m_true_axis[:, np.newaxis, np.newaxis]
bias_all_trials = m_estimated_all - m_true_broadcast

mean_bias = np.mean(np.abs(bias_all_trials), axis=-1)

worst_case_bias = np.max(np.abs(bias_all_trials), axis=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.875, 2.5), dpi=150, sharey=True)

pcm1 = ax1.pcolormesh(
    m_true_axis,
    distortion_amp_axis * 100,
    mean_bias.T,
    cmap=cmap_parula_r,
    shading="nearest",
    vmin=0,
    vmax=threshold,
)
ax1.set_title(r"Mean bias: $|E\,[\hat{m}] - m|$")
ax1.set_xlabel("True modulation depth $m$ (rad)")
ax1.set_ylabel("Distortion amplitude (%)")

pcm2 = ax2.pcolormesh(
    m_true_axis,
    distortion_amp_axis * 100,
    worst_case_bias.T,
    cmap=cmap_parula_r,
    shading="nearest",
    vmin=0,
    vmax=threshold,
)
ax2.set_title(r"Worst-case bias: max$| \hat{m} - m|$")
ax2.set_xlabel("True modulation depth, $m$ (rad)")

fig.subplots_adjust(right=0.85)
cbar_ax = fig.add_axes([0.87, 0.15, 0.03, 0.7])
cbar = fig.colorbar(pcm2, cax=cbar_ax)
cbar.set_label("Bias in $m$ (rad)")

plt.show()